## Setup and Imports

In [1]:
import torch
import torch.nn as nn
from src.transformer import MultiHeadAttention

# Set a seed for reproducibility
torch.manual_seed(42)

## Configuration and Initialization

Create a MultiHeadAttention instance with simple parameters to make debugging easier.

In [2]:
# Simple configuration for debugging
batch_size = 1
seq_len = 4
d_in = 8
d_out = 8
num_heads = 2

print("CONFIGURATION:")
print(f"  - Batch size: {batch_size}")
print(f"  - Sequence length: {seq_len}")
print(f"  - d_in: {d_in}")
print(f"  - d_out: {d_out}")
print(f"  - num_heads: {num_heads}")
print(f"  - head_dim: {d_out // num_heads}")

CONFIGURATION:
  - Batch size: 1
  - Sequence length: 4
  - d_in: 8
  - d_out: 8
  - num_heads: 2
  - head_dim: 4


In [3]:
# Create the module
torch.manual_seed(42)
mha = MultiHeadAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=10,
    dropout=0.0,  # No dropout for debugging
    num_heads=num_heads,
    qkv_bias=False
)
mha.eval()

print("✓ MultiHeadAttention module created")

✓ MultiHeadAttention module created


## Example Input

In [4]:
# Example input
torch.manual_seed(123)
x = torch.randn(batch_size, seq_len, d_in)

print("INPUT")
print(f"Shape: {x.shape}")
print(f"\nx[0, :, :3] (first 3 dimensions of each token):")
print(x[0, :, :3])

INPUT
Shape: torch.Size([1, 4, 8])

x[0, :, :3] (first 3 dimensions of each token):
tensor([[ 0.3374, -0.1778, -0.3035],
        [ 0.7671, -1.1925,  0.6984],
        [-0.0770, -1.0205, -0.1690],
        [ 0.4965, -1.5723,  0.9666]])


## STEP 1: Q, K, V Projections

The first step transforms the input into three representations: query, key, and value.

In [5]:
with torch.no_grad():
    queries = mha.W_query(x)
    keys = mha.W_key(x)
    values = mha.W_value(x)

print("STEP 1: Q, K, V Projections")
print(f"Queries shape: {queries.shape}")
print(f"Keys shape: {keys.shape}")
print(f"Values shape: {values.shape}")
print(f"\nQuery[0, 0, :] (first token):")
print(queries[0, 0, :])

STEP 1: Q, K, V Projections
Queries shape: torch.Size([1, 4, 8])
Keys shape: torch.Size([1, 4, 8])
Values shape: torch.Size([1, 4, 8])

Query[0, 0, :] (first token):
tensor([-0.1476,  0.1232, -0.0618, -0.0851,  0.2296, -0.2813,  0.3046, -0.0268])


## STEP 2: Multi-Head Reshaping

Reshape the tensors to separate the different attention heads.

In [6]:
head_dim = d_out // num_heads

# View: separate the heads
Q = queries.view(batch_size, seq_len, num_heads, head_dim)
K = keys.view(batch_size, seq_len, num_heads, head_dim)
V = values.view(batch_size, seq_len, num_heads, head_dim)

print("STEP 2: Multi-Head Reshaping")
print(f"\nAfter view:")
print(f"  Q: {queries.shape} -> {Q.shape}")
print(f"  K: {keys.shape} -> {K.shape}")
print(f"  V: {values.shape} -> {V.shape}")

# Transpose: move num_heads to the second position
Q = Q.transpose(1, 2)
K = K.transpose(1, 2)
V = V.transpose(1, 2)

print(f"\nAfter transposition:")
print(f"  Q: {Q.shape}")
print(f"  K: {K.shape}")
print(f"  V: {V.shape}")
print(f"  Format: (batch, num_heads, seq_len, head_dim)")

STEP 2: Multi-Head Reshaping

After view:
  Q: torch.Size([1, 4, 8]) -> torch.Size([1, 4, 2, 4])
  K: torch.Size([1, 4, 8]) -> torch.Size([1, 4, 2, 4])
  V: torch.Size([1, 4, 8]) -> torch.Size([1, 4, 2, 4])

After transposition:
  Q: torch.Size([1, 2, 4, 4])
  K: torch.Size([1, 2, 4, 4])
  V: torch.Size([1, 2, 4, 4])
  Format: (batch, num_heads, seq_len, head_dim)


## STEP 3: Attention Score Calculation

Calculate attention scores as the dot product between queries and keys.

In [7]:
# Prodotto scalare Q @ K^T
attn_scores = Q @ K.transpose(2, 3)

print("STEP 3: Attention Score Calculation")
print(f"Scores shape: {attn_scores.shape}")
print(f"  Format: (batch, num_heads, seq_len_q, seq_len_k)")
print(f"\nScores for head 0 (before scaling and masking):")
print(attn_scores[0, 0])

# Scaling
scaled_scores = attn_scores / (head_dim ** 0.5)
print(f"\nScores after scaling (÷ sqrt({head_dim})):")
print(scaled_scores[0, 0])

STEP 3: Attention Score Calculation
Scores shape: torch.Size([1, 2, 4, 4])
  Format: (batch, num_heads, seq_len_q, seq_len_k)

Scores for head 0 (before scaling and masking):
tensor([[ 0.0420,  0.0328,  0.0043, -0.1482],
        [ 0.2755,  0.5822,  0.0218, -0.5557],
        [ 0.3456,  0.7667, -0.2011, -0.5629],
        [ 0.6273,  0.0511,  0.1904, -0.6483]])

Scores after scaling (÷ sqrt(4)):
tensor([[ 0.0210,  0.0164,  0.0021, -0.0741],
        [ 0.1378,  0.2911,  0.0109, -0.2778],
        [ 0.1728,  0.3833, -0.1006, -0.2815],
        [ 0.3137,  0.0256,  0.0952, -0.3242]])


## STEP 4: Causal Mask Application

The causal mask prevents tokens from looking into the future.

In [8]:
# Get the mask
mask = mha.mask[:seq_len, :seq_len]

print("STEP 4: Causal Mask Application")
print(f"\nMask ({seq_len}x{seq_len}):")
print(mask)
print(f"\nLegend: 1 = masked (future), 0 = visible (past)")

# Apply the mask
mask_bool = mask.bool()
masked_scores = scaled_scores.clone()
masked_scores.masked_fill_(mask_bool, -torch.inf)

print(f"\nScores after masking (head 0):")
print(masked_scores[0, 0])
print(f"\nNote: -inf will become 0 after softmax")

STEP 4: Causal Mask Application

Mask (4x4):
tensor([[0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.]])

Legend: 1 = masked (future), 0 = visible (past)

Scores after masking (head 0):
tensor([[ 0.0210,    -inf,    -inf,    -inf],
        [ 0.1378,  0.2911,    -inf,    -inf],
        [ 0.1728,  0.3833, -0.1006,    -inf],
        [ 0.3137,  0.0256,  0.0952, -0.3242]])

Note: -inf will become 0 after softmax


## STEP 5: Softmax → Attention Weights

Convert scores into probabilities with softmax.

In [9]:
# Softmax
attn_weights = torch.softmax(masked_scores, dim=-1)

print("STEP 5: Softmax -> Attention Weights")
print(f"Weights shape: {attn_weights.shape}")
print(f"\nAttention weights for head 0:")
print(attn_weights[0, 0])

print(f"\nCheck: does every row sum to 1?")
for i in range(seq_len):
    row_sum = attn_weights[0, 0, i, :].sum().item()
    print(f"  Token {i}: sum = {row_sum:.6f}")

print(f"\nInterpretation:")
print(f"  - Row i = how much token i attends to other tokens")
print(f"  - Column j = how much token j is attended to")
print(f"  - Diagonal = self-attention")
print(f"  - Upper triangle = 0 (causal mask)")

STEP 5: Softmax -> Attention Weights
Weights shape: torch.Size([1, 2, 4, 4])

Attention weights for head 0:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4617, 0.5383, 0.0000, 0.0000],
        [0.3339, 0.4121, 0.2540, 0.0000],
        [0.3245, 0.2433, 0.2608, 0.1715]])

Check: does every row sum to 1?
  Token 0: sum = 1.000000
  Token 1: sum = 1.000000
  Token 2: sum = 1.000000
  Token 3: sum = 1.000000

Interpretation:
  - Row i = how much token i attends to other tokens
  - Column j = how much token j is attended to
  - Diagonal = self-attention
  - Upper triangle = 0 (causal mask)


## STEP 6: Context Vector (Weighted Sum)

Multiply attention weights by values to obtain the context.

In [10]:
# Weighted sum
context = attn_weights @ V

print("STEP 6: Context Vector (Weighted Sum)")
print(f"Context shape: {context.shape}")
print(f"  Format: (batch, num_heads, seq_len, head_dim)")
print(f"\nContext vector for token 0, head 0:")
print(context[0, 0, 0, :])

STEP 6: Context Vector (Weighted Sum)
Context shape: torch.Size([1, 2, 4, 4])
  Format: (batch, num_heads, seq_len, head_dim)

Context vector for token 0, head 0:
tensor([ 0.0853, -0.1247, -0.2917,  0.2840])


## STEP 7: Concatenating the Heads

Combine representations from all attention heads.

In [11]:
# Transposition
context_transposed = context.transpose(1, 2)

print("STEP 7: Concatenating the Heads")
print(f"After transposition: {context.shape} -> {context_transposed.shape}")

# Concatenate
context_concat = context_transposed.contiguous().view(batch_size, seq_len, d_out)
print(f"After concatenation: {context_concat.shape}")
print(f"  Format: (batch, seq_len, d_out)")

STEP 7: Concatenating the Heads
After transposition: torch.Size([1, 2, 4, 4]) -> torch.Size([1, 4, 2, 4])
After concatenation: torch.Size([1, 4, 8])
  Format: (batch, seq_len, d_out)


## STEP 8: Final Projection

Apply the final linear projection.

In [12]:
# Output projection
with torch.no_grad():
    output = mha.out_proj(context_concat)

print("STEP 8: Final Projection")
print(f"Output shape: {output.shape}")
print(f"\nOutput[0, 0, :] (first token):")
print(output[0, 0, :])

STEP 8: Final Projection
Output shape: torch.Size([1, 4, 8])

Output[0, 0, :] (first token):
tensor([ 0.1860, -0.4190,  0.0371,  0.0727, -0.0105, -0.4630,  0.1482, -0.3390])


## Verification: Comparison with Complete forward()

Verify that the manual decomposition produces the same result as the module's forward() method.

In [13]:
# Complete forward pass
with torch.no_grad():
    output_full = mha(x)

# Confronta
diff = (output - output_full).abs().max().item()

print("VERIFICATION: Comparison with complete forward()")
print(f"Maximum difference: {diff:.10f}")

if diff < 1e-6:
    print("✓ The manual decomposition is identical to forward()!")
else:
    print("✗ There is a difference (possible problem)")

VERIFICATION: Comparison with complete forward()
Maximum difference: 0.0000000000
✓ The manual decomposition is identical to forward()!


## Summary: Data Flow

Summary of transformations through the layer.

In [14]:
print("SUMMARY: Data Flow")
print("="*70)
print(f"1. Input:        {x.shape}")
print(f"2. Q, K, V:      {queries.shape}")
print(f"3. Separate heads: {Q.shape}")
print(f"4. Scores:       {attn_scores.shape}")
print(f"5. Weights:      {attn_weights.shape}")
print(f"6. Context:      {context.shape}")
print(f"7. Concatenation: {context_concat.shape}")
print(f"8. Output:       {output.shape}")
print("="*70)

SUMMARY: Data Flow
1. Input:        torch.Size([1, 4, 8])
2. Q, K, V:      torch.Size([1, 4, 8])
3. Separate heads: torch.Size([1, 2, 4, 4])
4. Scores:       torch.Size([1, 2, 4, 4])
5. Weights:      torch.Size([1, 2, 4, 4])
6. Context:      torch.Size([1, 2, 4, 4])
7. Concatenation: torch.Size([1, 4, 8])
8. Output:       torch.Size([1, 4, 8])


---

## Attention Weight Visualization

Display attention weights in table form for easier understanding.

In [15]:
# Visualization setup
mha_viz = MultiHeadAttention(
    d_in=16,
    d_out=16,
    context_length=20,
    dropout=0.0,
    num_heads=2,
    qkv_bias=False
)
mha_viz.eval()

# Input
seq_len_viz = 6
x_viz = torch.randn(1, seq_len_viz, 16)

print("VISUALIZATION: Attention Weights")
print(f"\nInput: {seq_len_viz} token")
print(f"Each token can attend only to itself and the past (causal mask)\n")

VISUALIZATION: Attention Weights

Input: 6 token
Each token can attend only to itself and the past (causal mask)



In [16]:
# Calculate weights manually
with torch.no_grad():
    Q_viz = mha_viz.W_query(x_viz)
    K_viz = mha_viz.W_key(x_viz)
    V_viz = mha_viz.W_value(x_viz)
    
    Q_viz = Q_viz.view(1, seq_len_viz, mha_viz.num_heads, mha_viz.head_dim).transpose(1, 2)
    K_viz = K_viz.view(1, seq_len_viz, mha_viz.num_heads, mha_viz.head_dim).transpose(1, 2)
    V_viz = V_viz.view(1, seq_len_viz, mha_viz.num_heads, mha_viz.head_dim).transpose(1, 2)
    
    scores_viz = Q_viz @ K_viz.transpose(2, 3)
    mask_viz = mha_viz.mask[:seq_len_viz, :seq_len_viz].bool()
    scores_viz.masked_fill_(mask_viz, -torch.inf)
    
    weights_viz = torch.softmax(scores_viz / (mha_viz.head_dim ** 0.5), dim=-1)

print("Attention weights for head 0:")
print("(Rows = query tokens, columns = key tokens)")
print("\n      ", end="")
for j in range(seq_len_viz):
    print(f"T{j}    ", end="")
print()

for i in range(seq_len_viz):
    print(f"T{i}:  ", end="")
    for j in range(seq_len_viz):
        val = weights_viz[0, 0, i, j].item()
        if val < 1e-6:
            print("  .   ", end="")
        else:
            print(f"{val:.2f}  ", end="")
    print()

print("\nLegend:")
print("  - T0, T1, ... = Token 0, 1, ...")
print("  - '.' = weight ~0 (masked)")
print("  - Numbers = attention weight")
print("  - Every row sums to 1.0")

Attention weights for head 0:
(Rows = query tokens, columns = key tokens)

      T0    T1    T2    T3    T4    T5    
T0:  1.00    .     .     .     .     .   
T1:  0.47  0.53    .     .     .     .   
T2:  0.38  0.34  0.28    .     .     .   
T3:  0.20  0.26  0.32  0.22    .     .   
T4:  0.18  0.18  0.19  0.25  0.20    .   
T5:  0.17  0.13  0.18  0.19  0.15  0.18  

Legend:
  - T0, T1, ... = Token 0, 1, ...
  - '.' = weight ~0 (masked)
  - Numbers = attention weight
  - Every row sums to 1.0
